In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

In [11]:
load_dotenv("../.env")
census_key = os.getenv("CENSUS_API_KEY")

In [12]:
years = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

all_dfs = []

In [13]:
for year in years:
    url = f"https://api.census.gov/data/{year}/acs/acs5/subject"

    params = {
        "get": "NAME,S1501_C02_008E,S1501_C02_009E,S1501_C02_015E",
        "for": "county:*",
        "in": "state:54",   # West Virginia only
        "key": census_key
    }

    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame(data[1:], columns=data[0])

    df = df.rename(columns={
        "S1501_C02_008E": "Pct_Less_Than_HS",
        "S1501_C02_009E": "Pct_HS_Grad",
        "S1501_C02_015E": "Pct_Bachelors_Plus"
    })

    df["year"] = year

    # Create full FIPS
    df["FIPS"] = df["state"] + df["county"]

    all_dfs.append(df)

education_panel = pd.concat(all_dfs, ignore_index=True)

education_panel

,NAME,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,state,county,year,FIPS
0,"Grant County, West Virginia",12.3,54.7,9.9,54,023,2013,54023
1,"Summers County, West Virginia",15.8,40.5,11.1,54,089,2013,54089
2,"Brooke County, West Virginia",5.3,49.4,15.0,54,009,2013,54009
3,"Greenbrier County, West Virginia",12.3,44.1,15.9,54,025,2013,54025
4,"Hardy County, West Virginia",16.7,51.8,7.5,54,031,2013,54031
...,...,...,...,...,...,...,...,...
600,"Webster County, West Virginia",9.5,49.2,12.6,54,101,2023,54101
601,"Wetzel County, West Virginia",7.5,48.6,13.5,54,103,2023,54103
602,"Wirt County, West Virginia",9.6,47.9,16.2,54,105,2023,54105
603,"Wood County, West Virginia",7.0,35.5,22.9,54,107,2023,54107


In [14]:
education_panel["FIPS_Code"] = education_panel["state"] + education_panel["county"]
education_panel.drop(columns=["state", "county", "FIPS"], inplace=True)
education_panel

,NAME,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,year,FIPS_Code
0,"Grant County, West Virginia",12.3,54.7,9.9,2013,54023
1,"Summers County, West Virginia",15.8,40.5,11.1,2013,54089
2,"Brooke County, West Virginia",5.3,49.4,15.0,2013,54009
3,"Greenbrier County, West Virginia",12.3,44.1,15.9,2013,54025
4,"Hardy County, West Virginia",16.7,51.8,7.5,2013,54031
...,...,...,...,...,...,...
600,"Webster County, West Virginia",9.5,49.2,12.6,2023,54101
601,"Wetzel County, West Virginia",7.5,48.6,13.5,2023,54103
602,"Wirt County, West Virginia",9.6,47.9,16.2,2023,54105
603,"Wood County, West Virginia",7.0,35.5,22.9,2023,54107


In [15]:
education_panel["County"] = education_panel["NAME"].apply(lambda x: x.split(",")[0])
education_panel.drop(columns=["NAME"], inplace=True)
education_panel

,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,year,FIPS_Code,County
0,12.3,54.7,9.9,2013,54023,Grant County
1,15.8,40.5,11.1,2013,54089,Summers County
2,5.3,49.4,15.0,2013,54009,Brooke County
3,12.3,44.1,15.9,2013,54025,Greenbrier County
4,16.7,51.8,7.5,2013,54031,Hardy County
...,...,...,...,...,...,...
600,9.5,49.2,12.6,2023,54101,Webster County
601,7.5,48.6,13.5,2023,54103,Wetzel County
602,9.6,47.9,16.2,2023,54105,Wirt County
603,7.0,35.5,22.9,2023,54107,Wood County


In [16]:
education_panel.rename(columns={"year": "Year"}, inplace=True)
education_panel

,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,Year,FIPS_Code,County
0,12.3,54.7,9.9,2013,54023,Grant County
1,15.8,40.5,11.1,2013,54089,Summers County
2,5.3,49.4,15.0,2013,54009,Brooke County
3,12.3,44.1,15.9,2013,54025,Greenbrier County
4,16.7,51.8,7.5,2013,54031,Hardy County
...,...,...,...,...,...,...
600,9.5,49.2,12.6,2023,54101,Webster County
601,7.5,48.6,13.5,2023,54103,Wetzel County
602,9.6,47.9,16.2,2023,54105,Wirt County
603,7.0,35.5,22.9,2023,54107,Wood County


In [17]:
education_panel = education_panel[["Year", "FIPS_Code", "County", "Pct_Less_Than_HS", "Pct_HS_Grad", "Pct_Bachelors_Plus"]]
education_panel

,Year,FIPS_Code,County,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus
0,2013,54023,Grant County,12.3,54.7,9.9
1,2013,54089,Summers County,15.8,40.5,11.1
2,2013,54009,Brooke County,5.3,49.4,15.0
3,2013,54025,Greenbrier County,12.3,44.1,15.9
4,2013,54031,Hardy County,16.7,51.8,7.5
...,...,...,...,...,...,...
600,2023,54101,Webster County,9.5,49.2,12.6
601,2023,54103,Wetzel County,7.5,48.6,13.5
602,2023,54105,Wirt County,9.6,47.9,16.2
603,2023,54107,Wood County,7.0,35.5,22.9


In [18]:
education_panel.to_csv("education_data.csv", index=False)